<a href="https://colab.research.google.com/github/shafi21001/fruit-freshness-edge-ai-study/blob/main/notebooks/07_android_export_executorch_xnnpack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ANDROID EXPORT ENVIRONMENT — INITIAL INSTALLATION
# Hardware: CPU runtime is sufficient
#
# Run only in a NEW Google Colab notebook.
# Do not run in the existing baseline notebook.
# ============================================================

import sys
import subprocess

print("Python executable:", sys.executable)
print("Python version:", sys.version)

# Install the official ExecuTorch package.
# Pip resolves its declared dependencies.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "executorch"
])

print("\nEXECUTORCH INSTALLATION COMMAND COMPLETED")

Python executable: /usr/bin/python3
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]

EXECUTORCH INSTALLATION COMMAND COMPLETED


In [ ]:
# ============================================================
# ANDROID EXPORT — CELL 2
# Hardware: CPU only
# Purpose: Verify installed package versions and XNNPACK APIs.
# No model loading, calibration, inference, or training.
# ============================================================

import sys
import importlib
import importlib.util
import importlib.metadata
import json

packages = ["torch", "torchvision", "torchao", "executorch"]

print("PYTHON:", sys.version)
print("\nPACKAGE VERSIONS")

versions = {}

for package in packages:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None

    print(f"{package}: {versions[package]}")

modules = [
    "executorch",
    "executorch.exir",
    "executorch.backends.xnnpack",
    "executorch.backends.xnnpack.quantizer.xnnpack_quantizer",
    "executorch.backends.xnnpack.partition.xnnpack_partitioner",
    "torchao.quantization.pt2e",
    "torchao.quantization.pt2e.quantize_pt2e",
]

print("\nMODULE AVAILABILITY")

module_status = {}

for name in modules:
    try:
        available = importlib.util.find_spec(name) is not None
    except Exception as exc:
        available = False
        print(name, "ERROR:", repr(exc))

    module_status[name] = available
    print(name, "->", available)

print("\nQUANTIZATION API CHECK")

api_status = {}

try:
    quantizer = importlib.import_module(
        "executorch.backends.xnnpack.quantizer.xnnpack_quantizer"
    )

    for name in [
        "XNNPACKQuantizer",
        "get_symmetric_quantization_config"
    ]:
        api_status[name] = hasattr(quantizer, name)
        print(name, "->", api_status[name])

except Exception as exc:
    api_status["quantizer_import_error"] = repr(exc)
    print("Quantizer import error:", repr(exc))

try:
    pt2e = importlib.import_module(
        "torchao.quantization.pt2e.quantize_pt2e"
    )

    for name in ["prepare_pt2e", "convert_pt2e"]:
        api_status[name] = hasattr(pt2e, name)
        print(name, "->", api_status[name])

except Exception as exc:
    api_status["pt2e_import_error"] = repr(exc)
    print("PT2E import error:", repr(exc))

result = {
    "python": sys.version,
    "versions": versions,
    "modules": module_status,
    "apis": api_status
}

print("\nVERIFICATION SUMMARY")
print(json.dumps(result, indent=2))

print("\nVERIFICATION COMPLETE")

PYTHON: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]

PACKAGE VERSIONS
torch: 2.15.0
torchvision: 0.29.1+cpu
torchao: 0.18.0
executorch: 1.6.0

MODULE AVAILABILITY
executorch -> True
executorch.exir -> True
executorch.backends.xnnpack -> True


W1126 18:49:21.472000 9648 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


executorch.backends.xnnpack.quantizer.xnnpack_quantizer -> True
executorch.backends.xnnpack.partition.xnnpack_partitioner -> True
torchao.quantization.pt2e -> True
torchao.quantization.pt2e.quantize_pt2e -> True

QUANTIZATION API CHECK
XNNPACKQuantizer -> True
get_symmetric_quantization_config -> True
prepare_pt2e -> True
convert_pt2e -> True

VERIFICATION SUMMARY
{
  "python": "3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]",
  "versions": {
    "torch": "2.15.0",
    "torchvision": "0.29.1+cpu",
    "torchao": "0.18.0",
    "executorch": "1.6.0"
  },
  "modules": {
    "executorch": true,
    "executorch.exir": true,
    "executorch.backends.xnnpack": true,
    "executorch.backends.xnnpack.quantizer.xnnpack_quantizer": true,
    "executorch.backends.xnnpack.partition.xnnpack_partitioner": true,
    "torchao.quantization.pt2e": true,
    "torchao.quantization.pt2e.quantize_pt2e": true
  },
  "apis": {
    "XNNPACKQuantizer": true,
    "get_symmetric_quantization_config": true,
   

In [ ]:
# ============================================================
# ANDROID EXPORT — CELL 3
# Hardware: CPU only
#
# Purpose:
#   Diagnose installed package dependency compatibility.
#   No installation, model loading, or dataset access.
# ============================================================

import sys
import json
import subprocess
import importlib.metadata as metadata

packages = ["torch", "torchvision", "torchao", "executorch"]

print("1. INSTALLED PACKAGE REQUIREMENTS")
requirements = {}

for package in packages:
    try:
        dist = metadata.distribution(package)
        requirements[package] = {
            "version": dist.version,
            "requires_dist": dist.metadata.get_all("Requires-Dist") or []
        }

        print(f"\n{package} {dist.version}")
        for requirement in requirements[package]["requires_dist"]:
            print("  ", requirement)

    except metadata.PackageNotFoundError:
        requirements[package] = None
        print(f"\n{package}: NOT INSTALLED")

print("\n\n2. PIP DEPENDENCY CHECK")

check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)

print("Return code:", check.returncode)
print("STDOUT:")
print(check.stdout or "(empty)")
print("STDERR:")
print(check.stderr or "(empty)")

print("\n\n3. TORCHAO OPERATOR AVAILABILITY")

import torch

operator_results = {}

for operator_name in [
    "dequantize_affine",
    "quantize_affine",
]:
    try:
        operator = getattr(torch.ops.torchao, operator_name)
        operator_results[operator_name] = {
            "available": True,
            "representation": str(operator)
        }
    except Exception as exc:
        operator_results[operator_name] = {
            "available": False,
            "error": repr(exc)
        }

    print(operator_name, ":", operator_results[operator_name])

result = {
    "python_version": sys.version,
    "package_requirements": requirements,
    "pip_check": {
        "return_code": check.returncode,
        "stdout": check.stdout,
        "stderr": check.stderr
    },
    "torchao_operators": operator_results,
    "environment_modified": False,
    "model_loaded": False,
    "calibration_performed": False,
    "inference_performed": False,
    "training_performed": False
}

print("\n\n4. DIAGNOSTIC SUMMARY")
print(json.dumps(result, indent=2))

print("\nDIAGNOSTIC COMPLETE")

1. INSTALLED PACKAGE REQUIREMENTS

torch 2.15.0
   filelock
   typing-extensions>=4.13.0
   setuptools>=77.0.3
   sympy>=1.13.3
   networkx>=2.5.1
   jinja2
   fsspec>=0.8.5
   cuda-toolkit[cublas,cudart,cufft,cufile,cupti,cusolver,cusparse,nvjitlink,nvrtc,nvtx]==13.2.2; platform_system == "Linux"
   cuda-bindings<14,>=13.0.3; platform_system == "Linux" and python_version < "3.15"
   nvidia-cudnn-cu13==9.26.0.51; platform_system == "Linux"
   nvidia-cusparselt-cu13==0.8.1; platform_system == "Linux"
   nvidia-nccl-cu13==2.30.7; platform_system == "Linux"
   nvidia-nvshmem-cu13==3.7.2; platform_system == "Linux"
   triton~=3.9.0; platform_system == "Linux" and python_version < "3.16"
   optree>=0.13.0; extra == "optree"
   opt-einsum>=3.3; extra == "opt-einsum"
   pyyaml; extra == "pyyaml"

torchvision 0.29.1+cpu
   numpy
   torch (>=2.14.0)
   pillow (!=8.3.*,>=5.3.0)
   gdown (>=4.7.3) ; extra == 'gdown'
   scipy ; extra == 'scipy'

torchao 0.18.0
   pytest==9.0.3; extra == "dev"
   u

In [ ]:
# ============================================================
# ANDROID EXPORT — CELL 4
# Hardware: CPU only
# Purpose: Inspect TorchAO imports and operator registration.
# No package changes, model loading, or dataset access.
# ============================================================

import torch
import importlib
import importlib.util
import pkgutil
import json

print("PyTorch:", torch.__version__)

# Step 1: Import TorchAO itself
print("\n1. TORCHAO TOP-LEVEL IMPORT")

try:
    import torchao
    print("TorchAO import: SUCCESS")
    print("TorchAO version:", getattr(torchao, "__version__", "not exposed"))
    print("TorchAO location:", torchao.__file__)
except Exception as exc:
    print("TorchAO import FAILED:", repr(exc))

# Step 2: Inspect available TorchAO modules
print("\n2. TORCHAO MODULES")

module_names = [
    "torchao._C",
    "torchao.ops",
    "torchao.quantization",
    "torchao.quantization.granularity",
    "torchao.quantization.pt2e",
    "torchao.quantization.pt2e.quantize_pt2e",
]

module_results = {}

for name in module_names:
    try:
        spec = importlib.util.find_spec(name)
        available = spec is not None
        error = None
    except Exception as exc:
        available = False
        error = repr(exc)

    module_results[name] = {
        "available": available,
        "error": error
    }
    print(name, "->", module_results[name])

# Step 3: Inspect the registered TorchAO operators
print("\n3. REGISTERED TORCHAO OPERATORS")

try:
    registered_ops = dir(torch.ops.torchao)
    print("Operator namespace entries:")
    print(registered_ops[:100])
except Exception as exc:
    registered_ops = []
    print("Namespace inspection failed:", repr(exc))

# Step 4: Attempt a direct import of TorchAO quantization
print("\n4. TORCHAO QUANTIZATION IMPORT")

try:
    quantization_module = importlib.import_module(
        "torchao.quantization"
    )
    print("Import: SUCCESS")
    print("Granularity available:",
          hasattr(quantization_module, "Granularity"))
    print("Module location:", quantization_module.__file__)
except Exception as exc:
    print("Import FAILED:", repr(exc))

# Step 5: Record results
result = {
    "pytorch_version": torch.__version__,
    "torchao_location": getattr(
        torchao, "__file__", None
    ) if "torchao" in globals() else None,
    "modules": module_results,
    "registered_torchao_namespace_entries": registered_ops,
    "environment_modified": False,
    "model_loaded": False,
    "calibration_performed": False,
    "inference_performed": False,
    "training_performed": False
}

print("\n5. DIAGNOSTIC SUMMARY")
print(json.dumps(result, indent=2))

print("\nTORCHAO REGISTRATION INSPECTION COMPLETE")

PyTorch: 2.15.0+cu132

1. TORCHAO TOP-LEVEL IMPORT
TorchAO import: SUCCESS
TorchAO version: 0.18.0
TorchAO location: /usr/local/lib/python3.13/dist-packages/torchao/__init__.py

2. TORCHAO MODULES
torchao._C -> {'available': False, 'error': None}
torchao.ops -> {'available': True, 'error': None}
torchao.quantization -> {'available': True, 'error': None}
torchao.quantization.granularity -> {'available': True, 'error': None}
torchao.quantization.pt2e -> {'available': True, 'error': None}
torchao.quantization.pt2e.quantize_pt2e -> {'available': True, 'error': None}

3. REGISTERED TORCHAO OPERATORS
Operator namespace entries:
['__doc__', '__loader__', '__name__', '__package__', '__spec__', '_dir', 'choose_qparams_affine', 'choose_scale_float8', 'dequantize_affine', 'dequantize_affine_float8_non_decomposed', 'name', 'quantize_affine', 'quantize_affine_float8_non_decomposed', 'reference_dqlinear_int4']

4. TORCHAO QUANTIZATION IMPORT
Import: SUCCESS
Granularity available: True
Module locatio

In [ ]:
import sys
import importlib.metadata as metadata

print("Python executable:", sys.executable)
print("Python version:", sys.version)

for package in ["torch", "torchvision", "torchao", "executorch"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python executable: /usr/bin/python3
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
torch: 2.15.0
torchvision: 0.29.1+cpu
torchao: 0.18.0
executorch: 1.6.0


In [ ]:
import torch
import torchao
import importlib

print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)
print("TorchAO location:", torchao.__file__)

print("\n1. TorchAO operator namespace")

try:
    print(dir(torch.ops.torchao))
except Exception as exc:
    print("ERROR:", repr(exc))

print("\n2. PT2E API imports")

try:
    from torchao.quantization.pt2e.quantize_pt2e import (
        prepare_pt2e,
        convert_pt2e
    )
    print("prepare_pt2e: AVAILABLE")
    print("convert_pt2e: AVAILABLE")
except Exception as exc:
    print("PT2E IMPORT FAILED:", repr(exc))

print("\n3. XNNPACK quantizer import")

try:
    from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
        XNNPACKQuantizer,
        get_symmetric_quantization_config
    )
    print("XNNPACKQuantizer: AVAILABLE")
    print("get_symmetric_quantization_config: AVAILABLE")
except Exception as exc:
    print("XNNPACK IMPORT FAILED:", repr(exc))

print("\n4. XNNPACK partitioner import")

try:
    from executorch.backends.xnnpack.partition.xnnpack_partitioner import (
        XnnpackPartitioner
    )
    print("XnnpackPartitioner: AVAILABLE")
except Exception as exc:
    print("PARTITIONER IMPORT FAILED:", repr(exc))

print("\nIMPORT DIAGNOSTIC COMPLETE")

PyTorch: 2.15.0+cu132
TorchAO: 0.18.0
TorchAO location: /usr/local/lib/python3.13/dist-packages/torchao/__init__.py

1. TorchAO operator namespace
['__doc__', '__loader__', '__name__', '__package__', '__spec__', '_dir', 'choose_qparams_affine', 'choose_scale_float8', 'dequantize_affine', 'dequantize_affine_float8_non_decomposed', 'name', 'quantize_affine', 'quantize_affine_float8_non_decomposed', 'reference_dqlinear_int4']

2. PT2E API imports
prepare_pt2e: AVAILABLE
convert_pt2e: AVAILABLE

3. XNNPACK quantizer import
XNNPACKQuantizer: AVAILABLE
get_symmetric_quantization_config: AVAILABLE

4. XNNPACK partitioner import
XnnpackPartitioner: AVAILABLE

IMPORT DIAGNOSTIC COMPLETE


In [ ]:
# ============================================================
# ANDROID EXPORT — CELL 6
# FP32 TORCH.EXPORT SMOKE TEST
#
# Hardware: CPU only
# Purpose:
#   Load original baseline checkpoint safely,
#   reconstruct MobileNetV3-Small,
#   and test torch.export using a dummy input.
#
# No calibration, quantization, dataset access,
# training, validation, or test evaluation.
# ============================================================

import json
import torch
import torchvision
from pathlib import Path
from torchvision.models import mobilenet_v3_small

ROOT = Path("/content/drive/MyDrive/Fruit_Freshness_Research")

EXPERIMENT = (
    ROOT / "experiments" / "baseline_mobilenetv3_small"
)

CHECKPOINT_PATH = (
    EXPERIMENT / "checkpoints" / "best_validation_macro_f1.pt"
)

OUTPUT_DIR = EXPERIMENT / "android_export"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSON = (
    OUTPUT_DIR / "fp32_torch_export_smoke_test_v1.json"
)

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Device: CPU")
print("Checkpoint:", CHECKPOINT_PATH)

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

# ------------------------------------------------------------
# 1. Safely load original checkpoint on CPU
# ------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

print("\nCheckpoint keys:", list(checkpoint.keys()))

completed_epoch = checkpoint.get("completed_epoch")
best_macro_f1 = checkpoint.get("best_macro_f1")

print("Completed epoch:", completed_epoch)
print("Best validation Macro-F1:", best_macro_f1)

# ------------------------------------------------------------
# 2. Reconstruct original model architecture
# ------------------------------------------------------------

model = mobilenet_v3_small(
    weights=None,
    num_classes=21
)

load_result = model.load_state_dict(
    checkpoint["model_state"],
    strict=True
)

model.eval()
model.cpu()

print("\nStrict checkpoint loading: PASSED")
print("Model device:", next(model.parameters()).device)

# ------------------------------------------------------------
# 3. Create a dummy input (NOT a dataset image)
# ------------------------------------------------------------

dummy_input = torch.randn(
    1, 3, 224, 224,
    dtype=torch.float32
)

# ------------------------------------------------------------
# 4. Test FP32 model forward pass
# ------------------------------------------------------------

with torch.inference_mode():
    output = model(dummy_input)

print("\nFP32 forward test")
print("Output shape:", tuple(output.shape))
print("Output dtype:", output.dtype)
print("Output finite:", bool(torch.isfinite(output).all()))

assert output.shape == (1, 21)
assert torch.isfinite(output).all()

# ------------------------------------------------------------
# 5. Export using torch.export
# ------------------------------------------------------------

print("\nStarting torch.export...")

exported_program = torch.export.export(
    model,
    (dummy_input,),
    strict=True
)

print("torch.export: PASSED")

# ------------------------------------------------------------
# 6. Inspect exported graph
# ------------------------------------------------------------

graph_text = str(exported_program.graph)

print("\nExported graph node count:",
      len(list(exported_program.graph.nodes)))

print("\nGraph preview:")
print(graph_text[:2000])

# ------------------------------------------------------------
# 7. Save export-test artifact
# ------------------------------------------------------------

exported_program_path = (
    OUTPUT_DIR / "mobilenetv3_small_fp32_export_test_v1.pt2"
)

torch.export.save(
    exported_program,
    exported_program_path
)

# ------------------------------------------------------------
# 8. Save reproducible evidence
# ------------------------------------------------------------

result = {
    "purpose": "FP32 torch.export smoke test for Android deployment workflow",
    "device": "cpu",
    "pytorch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "completed_epoch": completed_epoch,
    "best_validation_macro_f1": best_macro_f1,
    "architecture": "MobileNetV3-Small",
    "num_classes": 21,
    "input_shape": [1, 3, 224, 224],
    "input_dtype": "float32",
    "strict_checkpoint_loading": True,
    "forward_pass_passed": True,
    "output_shape": list(output.shape),
    "output_dtype": str(output.dtype),
    "output_finite": bool(torch.isfinite(output).all()),
    "torch_export_passed": True,
    "exported_graph_node_count": len(
        list(exported_program.graph.nodes)
    ),
    "exported_program_path": str(exported_program_path),
    "calibration_performed": False,
    "quantization_performed": False,
    "training_performed": False,
    "dataset_accessed": False,
    "validation_accessed": False,
    "test_accessed": False,
    "gpu_used": False
}

with open(OUTPUT_JSON, "w") as f:
    json.dump(result, f, indent=2)

print("\nEvidence saved:", OUTPUT_JSON)
print("Exported test artifact:", exported_program_path)
print("\nFP32 EXPORT SMOKE TEST COMPLETE")

PyTorch: 2.15.0+cu132
Torchvision: 0.29.1+cpu
Device: CPU
Checkpoint: /content/drive/MyDrive/Fruit_Freshness_Research/experiments/baseline_mobilenetv3_small/checkpoints/best_validation_macro_f1.pt


AssertionError: Checkpoint not found: /content/drive/MyDrive/Fruit_Freshness_Research/experiments/baseline_mobilenetv3_small/checkpoints/best_validation_macro_f1.pt

In [ ]:
# ============================================================
# ANDROID EXPORT — CELL 7
# CPU only
# Purpose: Locate the original baseline checkpoint.
# Read-only file inspection.
# ============================================================

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Fruit_Freshness_Research")

print("1. Drive root exists:", Path("/content/drive").exists())
print("2. Research root exists:", ROOT.exists())

if ROOT.exists():
    print("\nTop-level research folders:")
    for item in sorted(ROOT.iterdir()):
        print(" -", item.name, "(directory)" if item.is_dir() else "(file)")

EXP = ROOT / "experiments" / "baseline_mobilenetv3_small"

print("\n3. Experiment folder exists:", EXP.exists())

if EXP.exists():
    print("\nExperiment contents:")
    for item in sorted(EXP.iterdir()):
        print(" -", item.name, "(directory)" if item.is_dir() else "(file)")

    CHECKPOINT_DIR = EXP / "checkpoints"
    print("\n4. Checkpoints directory exists:", CHECKPOINT_DIR.exists())

    if CHECKPOINT_DIR.exists():
        print("\nCheckpoint files:")
        for item in sorted(CHECKPOINT_DIR.iterdir()):
            print(f" - {item.name} | {item.stat().st_size:,} bytes")

# Search only the experiment folder for likely checkpoint names.
# This checks filenames only, not checkpoint contents.
print("\n5. Checkpoint filename search:")

if EXP.exists():
    matches = []
    for path in EXP.rglob("*"):
        if path.is_file() and (
            "best" in path.name.lower()
            and path.suffix in [".pt", ".pth", ".ckpt"]
        ):
            matches.append(path)

    if matches:
        for path in matches:
            print("FOUND:", path)
    else:
        print("No matching checkpoint files found.")
else:
    print("Experiment directory unavailable.")

print("\nFILE LOCATION CHECK COMPLETE")

1. Drive root exists: True
2. Research root exists: True

Top-level research folders:
 - experiments (directory)

3. Experiment folder exists: True

Experiment contents:
 - android_export (directory)

4. Checkpoints directory exists: False

5. Checkpoint filename search:
No matching checkpoint files found.

FILE LOCATION CHECK COMPLETE


In [ ]:
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")
RESEARCH_ROOT = MYDRIVE / "Fruit_Freshness_Research"

print("Searching MyDrive for baseline checkpoint files...")
print("This is a filename-only search (no image/model loading).\n")

matches = []

for path in MYDRIVE.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".pt", ".pth", ".ckpt"}:
        name = path.name.lower()
        if any(term in name for term in ["best", "checkpoint", "mobilenet"]):
            matches.append(path)

print(f"Checkpoint candidates found: {len(matches)}\n")

for i, path in enumerate(matches, 1):
    try:
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{i}. {path}")
        print(f"   Size: {size_mb:.2f} MB")
    except Exception:
        print(f"{i}. {path}")

print("\nCHECKPOINT SEARCH COMPLETE")

Searching MyDrive for baseline checkpoint files...
This is a filename-only search (no image/model loading).

Checkpoint candidates found: 0


CHECKPOINT SEARCH COMPLETE


In [ ]:
from pathlib import Path
import hashlib

checkpoint = Path(
    "/content/drive/MyDrive/Fruit_Freshness_Research/"
    "experiments/baseline_mobilenetv3_small/"
    "checkpoints/best_validation_macro_f1.pt"
)

print("Checkpoint exists:", checkpoint.exists())

if checkpoint.exists():
    print("Checkpoint path:", checkpoint)
    print("File size:", f"{checkpoint.stat().st_size:,}", "bytes")

    sha256 = hashlib.sha256()
    with open(checkpoint, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    print("SHA-256:", sha256.hexdigest())
else:
    print("Checkpoint not found at this path.")

print("\nORIGINAL CHECKPOINT VERIFICATION COMPLETE")

Checkpoint exists: False
Checkpoint not found at this path.

ORIGINAL CHECKPOINT VERIFICATION COMPLETE


In [ ]:
from pathlib import Path

MYDRIVE = Path("/content/drive/MyDrive")

extensions = {".pt", ".pth", ".ckpt", ".safetensors", ".onnx"}

print("Searching all MyDrive for model artifacts...\n")

found = []

for path in MYDRIVE.rglob("*"):
    if path.is_file() and path.suffix.lower() in extensions:
        found.append(path)

print(f"Total model artifacts found: {len(found)}\n")

for i, path in enumerate(found, 1):
    try:
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{i}. {path}")
        print(f"   Size: {size_mb:.2f} MB")
    except OSError:
        print(f"{i}. {path} (unable to read size)")

print("\nFULL MODEL ARTIFACT SEARCH COMPLETE")

Searching all MyDrive for model artifacts...

Total model artifacts found: 0


FULL MODEL ARTIFACT SEARCH COMPLETE


In [5]:
from google.colab import drive
from pathlib import Path

# Refresh the Google Drive mount
drive.mount("/content/drive", force_remount=True)

# Check the exact checkpoint folder
checkpoint_dir = Path(
    "/content/drive/MyDrive/Fruit_Freshness_Research/"
    "experiments/baseline_mobilenetv3_small/checkpoints"
)

print("\nCheckpoint folder exists:", checkpoint_dir.exists())

if checkpoint_dir.exists():
    print("\nFiles visible to Colab:")
    for f in sorted(checkpoint_dir.iterdir()):
        if f.is_file():
            print(f"{f.name} | {f.stat().st_size:,} bytes")
else:
    print("\nFolder is still not visible to this Colab Drive mount.")

print("\nDRIVE REMOUNT CHECK COMPLETE")

Mounted at /content/drive

Checkpoint folder exists: True

Files visible to Colab:
best_validation_macro_f1.pt | 6,292,607 bytes
latest_checkpoint.pt | 18,774,425 bytes

DRIVE REMOUNT CHECK COMPLETE


In [2]:
# ============================================================
# ANDROID EXPORT — CELL 8
# CPU ONLY
# Purpose:
# 1. Load the recovered original baseline checkpoint
# 2. Reconstruct the MobileNetV3-Small architecture
# 3. Verify FP32 inference
# 4. Perform a torch.export smoke test
# 5. Save the exported artifact without overwriting existing files
# ============================================================

from pathlib import Path
import torch
import json
import hashlib
from torchvision.models import mobilenet_v3_small

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
ROOT = Path("/content/drive/MyDrive/Fruit_Freshness_Research")

CHECKPOINT_PATH = (
    ROOT / "experiments" / "baseline_mobilenetv3_small"
    / "checkpoints" / "best_validation_macro_f1.pt"
)

EXPORT_DIR = (
    ROOT / "experiments" / "baseline_mobilenetv3_small"
    / "android_export"
)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

assert CHECKPOINT_PATH.exists(), (
    f"Checkpoint not found: {CHECKPOINT_PATH}"
)

# Prevent accidental overwriting
EXPORT_PATH = EXPORT_DIR / "baseline_fp32_exported_program.pt2"
METADATA_PATH = EXPORT_DIR / "baseline_fp32_export_metadata.json"

assert not EXPORT_PATH.exists(), (
    f"Export file already exists: {EXPORT_PATH}"
)
assert not METADATA_PATH.exists(), (
    f"Metadata file already exists: {METADATA_PATH}"
)

print("Checkpoint found:", CHECKPOINT_PATH)
print("Checkpoint size:", f"{CHECKPOINT_PATH.stat().st_size:,}", "bytes")

# ------------------------------------------------------------
# 2. Compute checkpoint SHA-256
# ------------------------------------------------------------
sha256 = hashlib.sha256()

with open(CHECKPOINT_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

checkpoint_hash = sha256.hexdigest()

print("Checkpoint SHA-256:", checkpoint_hash)

# ------------------------------------------------------------
# 3. Load checkpoint safely on CPU
# ------------------------------------------------------------
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

print("\nCheckpoint keys:", list(checkpoint.keys()))

assert "model_state" in checkpoint, (
    "Expected 'model_state' key is missing."
)

# ------------------------------------------------------------
# 4. Reconstruct original model architecture
# ------------------------------------------------------------
NUM_CLASSES = 21

model = mobilenet_v3_small(
    weights=None,
    num_classes=NUM_CLASSES
)

model.load_state_dict(
    checkpoint["model_state"],
    strict=True
)

model.eval()
model.cpu()

print("\nModel reconstructed successfully.")
print("Architecture: MobileNetV3-Small")
print("Classes:", NUM_CLASSES)
print("Device: CPU")
print("Mode: eval")

# ------------------------------------------------------------
# 5. FP32 inference smoke test
# ------------------------------------------------------------
dummy_input = torch.randn(
    1, 3, 224, 224,
    dtype=torch.float32
)

with torch.inference_mode():
    output = model(dummy_input)

assert output.shape == (1, NUM_CLASSES), (
    f"Unexpected output shape: {output.shape}"
)

assert torch.isfinite(output).all(), (
    "Model output contains NaN or Inf."
)

print("\nFP32 inference smoke test: PASSED")
print("Output shape:", tuple(output.shape))
print("Output dtype:", output.dtype)
print("Finite outputs:", bool(torch.isfinite(output).all()))

# ------------------------------------------------------------
# 6. torch.export smoke test
# ------------------------------------------------------------
print("\nStarting torch.export...")

exported_program = torch.export.export(
    model,
    (dummy_input,),
    strict=True
)

print("torch.export: PASSED")

# ------------------------------------------------------------
# 7. Save export artifact
# ------------------------------------------------------------
torch.export.save(
    exported_program,
    EXPORT_PATH
)

# ------------------------------------------------------------
# 8. Save metadata
# ------------------------------------------------------------
metadata = {
    "model": "MobileNetV3-Small",
    "num_classes": NUM_CLASSES,
    "input_shape": [1, 3, 224, 224],
    "input_dtype": "float32",
    "checkpoint_filename": CHECKPOINT_PATH.name,
    "checkpoint_sha256": checkpoint_hash,
    "export_format": "torch.export ExportedProgram",
    "export_artifact": EXPORT_PATH.name,
    "device": "cpu",
    "training_performed": False,
    "dataset_accessed": False,
    "test_set_accessed": False,
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# 9. Verify saved artifacts
# ------------------------------------------------------------
print("\n--- SAVED ARTIFACTS ---")

for path in [EXPORT_PATH, METADATA_PATH]:
    assert path.exists()
    print(f"{path.name} | {path.stat().st_size:,} bytes")

print("\n" + "=" * 55)
print("FP32 CHECKPOINT RECOVERY + EXPORT SMOKE TEST COMPLETE")
print("=" * 55)

Checkpoint found: /content/drive/MyDrive/Fruit_Freshness_Research/experiments/baseline_mobilenetv3_small/checkpoints/best_validation_macro_f1.pt
Checkpoint size: 6,292,607 bytes
Checkpoint SHA-256: 327f216c7b5f911508a38abd34a30eae612cdc426c2828941cee9b2018f110cc

Checkpoint keys: ['completed_epoch', 'model_state', 'best_macro_f1', 'config_path', 'seed']

Model reconstructed successfully.
Architecture: MobileNetV3-Small
Classes: 21
Device: CPU
Mode: eval

FP32 inference smoke test: PASSED
Output shape: (1, 21)
Output dtype: torch.float32
Finite outputs: True

Starting torch.export...
torch.export: PASSED

--- SAVED ARTIFACTS ---
baseline_fp32_exported_program.pt2 | 7,430,390 bytes
baseline_fp32_export_metadata.json | 504 bytes

FP32 CHECKPOINT RECOVERY + EXPORT SMOKE TEST COMPLETE


In [8]:
# ============================================================
# ANDROID EXPORT — CELL 9 (CORRECTED)
# CPU ONLY
# Purpose: Reload the saved ExportedProgram and compare its
# outputs against the original FP32 model.
# ============================================================

from pathlib import Path
import torch
import json

ROOT = Path("/content/drive/MyDrive/Fruit_Freshness_Research")

EXPORT_DIR = (
    ROOT / "experiments" / "baseline_mobilenetv3_small"
    / "android_export"
)

EXPORT_PATH = EXPORT_DIR / "baseline_fp32_exported_program.pt2"
METADATA_PATH = EXPORT_DIR / "baseline_fp32_export_metadata.json"

# ------------------------------------------------------------
# 1. Confirm files
# ------------------------------------------------------------
assert EXPORT_PATH.exists(), f"Missing export: {EXPORT_PATH}"
assert METADATA_PATH.exists(), f"Missing metadata: {METADATA_PATH}"

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Export artifact:", EXPORT_PATH.name)
print("Artifact size:", f"{EXPORT_PATH.stat().st_size:,} bytes")
print("Metadata model:", metadata["model"])
print("Metadata input shape:", metadata["input_shape"])

# ------------------------------------------------------------
# 2. Load saved ExportedProgram
# ------------------------------------------------------------
print("\nReloading saved ExportedProgram...")

loaded_exported_program = torch.export.load(EXPORT_PATH)

# NOTE: Do not call .eval() on this wrapper.
# ExportedProgram.module() has a specialized interface.
exported_model = loaded_exported_program.module()

print("Reload: PASSED")

# ------------------------------------------------------------
# 3. Deterministic synthetic input
# ------------------------------------------------------------
torch.manual_seed(20261119)

test_input = torch.randn(
    1, 3, 224, 224,
    dtype=torch.float32
)

# ------------------------------------------------------------
# 4. Compare original and reloaded exported model
# ------------------------------------------------------------
with torch.inference_mode():
    original_output = model(test_input)
    exported_output = exported_model(test_input)

assert original_output.shape == exported_output.shape
assert torch.isfinite(original_output).all()
assert torch.isfinite(exported_output).all()

difference = (original_output - exported_output).abs()

max_abs_diff = difference.max().item()
mean_abs_diff = difference.mean().item()

outputs_close = torch.allclose(
    original_output,
    exported_output,
    rtol=1e-4,
    atol=1e-5
)

print("\n--- NUMERICAL COMPARISON ---")
print("Original output shape:", tuple(original_output.shape))
print("Exported output shape:", tuple(exported_output.shape))
print("Maximum absolute difference:", max_abs_diff)
print("Mean absolute difference:", mean_abs_diff)
print("Allclose (rtol=1e-4, atol=1e-5):", outputs_close)

assert outputs_close, (
    "Exported output differs from original model beyond tolerance."
)

# ------------------------------------------------------------
# 5. Save verification evidence (version-safe)
# ------------------------------------------------------------
VERIFICATION_PATH = (
    EXPORT_DIR / "fp32_export_reload_verification_v1.json"
)

if VERIFICATION_PATH.exists():
    VERIFICATION_PATH = (
        EXPORT_DIR / "fp32_export_reload_verification_v2.json"
    )

assert not VERIFICATION_PATH.exists(), (
    f"Refusing to overwrite existing evidence: {VERIFICATION_PATH}"
)

verification = {
    "verification": "saved_export_reload_equivalence",
    "export_artifact": EXPORT_PATH.name,
    "synthetic_input_shape": list(test_input.shape),
    "synthetic_input_dtype": str(test_input.dtype),
    "max_absolute_difference": max_abs_diff,
    "mean_absolute_difference": mean_abs_diff,
    "allclose": bool(outputs_close),
    "rtol": 1e-4,
    "atol": 1e-5,
    "device": "cpu",
    "dataset_accessed": False,
    "validation_accessed": False,
    "test_accessed": False,
    "training_performed": False,
}

with open(VERIFICATION_PATH, "w", encoding="utf-8") as f:
    json.dump(verification, f, indent=2)

print("\nVerification evidence saved:")
print(VERIFICATION_PATH)
print("\nFP32 EXPORT RELOAD VERIFICATION COMPLETE")

Export artifact: baseline_fp32_exported_program.pt2
Artifact size: 7,430,390 bytes
Metadata model: MobileNetV3-Small
Metadata input shape: [1, 3, 224, 224]

Reloading saved ExportedProgram...
Reload: PASSED

--- NUMERICAL COMPARISON ---
Original output shape: (1, 21)
Exported output shape: (1, 21)
Maximum absolute difference: 0.0
Mean absolute difference: 0.0
Allclose (rtol=1e-4, atol=1e-5): True

Verification evidence saved:
/content/drive/MyDrive/Fruit_Freshness_Research/experiments/baseline_mobilenetv3_small/android_export/fp32_export_reload_verification_v1.json

FP32 EXPORT RELOAD VERIFICATION COMPLETE


In [9]:
# ============================================================
# ANDROID EXPORT — CELL 10
# CPU ONLY
# Purpose: Verify calibration subset manifests and export artifacts.
# This cell does NOT load or decode images.
# ============================================================

from pathlib import Path
import json
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Fruit_Freshness_Research")

EXP_DIR = ROOT / "experiments" / "baseline_mobilenetv3_small"
ANDROID_DIR = EXP_DIR / "android_export"

CALIBRATION_CSV = (
    EXP_DIR / "quantization"
    / "calibration_train_manifest_630_v1.csv"
)

CALIBRATION_EVIDENCE = (
    EXP_DIR / "evidence"
    / "calibration_subset_selection_v1.json"
)

FP32_EXPORT = ANDROID_DIR / "baseline_fp32_exported_program.pt2"
FP32_METADATA = ANDROID_DIR / "baseline_fp32_export_metadata.json"
FP32_VERIFICATION = (
    ANDROID_DIR / "fp32_export_reload_verification_v1.json"
)

# ------------------------------------------------------------
# 1. Check required artifacts
# ------------------------------------------------------------
artifacts = {
    "Calibration CSV": CALIBRATION_CSV,
    "Calibration selection evidence": CALIBRATION_EVIDENCE,
    "FP32 export": FP32_EXPORT,
    "FP32 export metadata": FP32_METADATA,
    "FP32 reload verification": FP32_VERIFICATION,
}

print("=== ARTIFACT EXISTENCE CHECK ===")

for name, path in artifacts.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name}: {status}")
    if path.exists() and path.is_file():
        print(f"  Size: {path.stat().st_size:,} bytes")

# ------------------------------------------------------------
# 2. Read calibration manifest (metadata only)
# ------------------------------------------------------------
if CALIBRATION_CSV.exists():

    df_cal = pd.read_csv(CALIBRATION_CSV)

    print("\n=== CALIBRATION MANIFEST ===")
    print("Rows:", len(df_cal))
    print("Columns:", list(df_cal.columns))

    print("\nFirst 5 rows:")
    display(df_cal.head())

    # Display class balance for likely class/label columns
    possible_class_columns = [
        col for col in df_cal.columns
        if any(term in col.lower() for term in ["class", "label", "grade"])
    ]

    if possible_class_columns:
        for col in possible_class_columns:
            print(f"\nValue counts for '{col}':")
            print(df_cal[col].value_counts().sort_index())

else:
    print("\nCalibration manifest unavailable; skipping CSV inspection.")

# ------------------------------------------------------------
# 3. Inspect calibration selection evidence
# ------------------------------------------------------------
if CALIBRATION_EVIDENCE.exists():

    with open(CALIBRATION_EVIDENCE, "r", encoding="utf-8") as f:
        calibration_evidence = json.load(f)

    print("\n=== CALIBRATION SELECTION EVIDENCE ===")
    print(json.dumps(calibration_evidence, indent=2))

else:
    print("\nCalibration selection evidence JSON unavailable.")

# ------------------------------------------------------------
# 4. Inspect FP32 export verification evidence
# ------------------------------------------------------------
if FP32_VERIFICATION.exists():

    with open(FP32_VERIFICATION, "r", encoding="utf-8") as f:
        fp32_verification = json.load(f)

    print("\n=== FP32 EXPORT VERIFICATION EVIDENCE ===")
    print(json.dumps(fp32_verification, indent=2))

print("\nCELL 10 — CALIBRATION/EXPORT ARTIFACT AUDIT COMPLETE")

=== ARTIFACT EXISTENCE CHECK ===
Calibration CSV: FOUND
  Size: 503,768 bytes
Calibration selection evidence: FOUND
  Size: 1,565 bytes
FP32 export: FOUND
  Size: 7,430,390 bytes
FP32 export metadata: FOUND
  Size: 504 bytes
FP32 reload verification: FOUND
  Size: 484 bytes

=== CALIBRATION MANIFEST ===
Rows: 630
Columns: ['dataset', 'filename', 'relative_path', 'fruit', 'grade', 'file_size', 'sha256', 'duplicate_group_id', 'duplicate_group_unique_hashes', 'quarantine_reason', 'dataset_status', 'group_requires_quarantine', 'split_group_id', 'split', 'final_group_type', 'final_dataset_status', 'final_quarantine_flag', 'final_grouping_method', 'metadata_reconciliation_note', 'class_name', 'cache_path', 'calibration_image_path']

First 5 rows:


,dataset,filename,relative_path,fruit,grade,file_size,sha256,duplicate_group_id,duplicate_group_unique_hashes,quarantine_reason,...,split_group_id,split,final_group_type,final_dataset_status,final_quarantine_flag,final_grouping_method,metadata_reconciliation_note,class_name,cache_path,calibration_image_path
0,AFruitDB,IMG_20230930_213927.jpg,Fruits Grade/Fruit grade/Apple/1st grade/IMG_2...,Apple,1st grade,985459,0a3850fd7f0258c769cede51f9be4b2298be73c55490ec...,SINGLETON,1,NaN,...,SHA_0a3850fd7f0258c769cede51f9be4b2298be73c554...,train,SHA_SINGLETON_GROUP,RETAINED_QC_CLEAN,False,SHA256_SINGLE_IMAGE_GROUP,Legacy audit fields preserved for traceability...,Apple_1st grade,/content/fruit_freshness_cache/images/0a3850fd...,/content/drive/MyDrive/Fruit_Freshness_Researc...
1,AFruitDB,IMG_20230930_214245.jpg,Fruits Grade/Fruit grade/Apple/1st grade/IMG_2...,Apple,1st grade,809148,0f0a7ced697563e9b2ad78510e00e468182e74640d44d6...,SINGLETON,1,NaN,...,SHA_0f0a7ced697563e9b2ad78510e00e468182e74640d...,train,SHA_SINGLETON_GROUP,RETAINED_QC_CLEAN,False,SHA256_SINGLE_IMAGE_GROUP,Legacy audit fields preserved for traceability...,Apple_1st grade,/content/fruit_freshness_cache/images/0f0a7ced...,/content/drive/MyDrive/Fruit_Freshness_Researc...
2,AFruitDB,IMG_20230930_213747.jpg,Fruits Grade/Fruit grade/Apple/1st grade/IMG_2...,Apple,1st grade,1403835,14e34a5184960b7b12c5d2435bfef24d426614c80dc171...,SINGLETON,1,NaN,...,SHA_14e34a5184960b7b12c5d2435bfef24d426614c80d...,train,SHA_SINGLETON_GROUP,RETAINED_QC_CLEAN,False,SHA256_SINGLE_IMAGE_GROUP,Legacy audit fields preserved for traceability...,Apple_1st grade,/content/fruit_freshness_cache/images/14e34a51...,/content/drive/MyDrive/Fruit_Freshness_Researc...
3,AFruitDB,IMG_20230930_212759.jpg,Fruits Grade/Fruit grade/Apple/1st grade/IMG_2...,Apple,1st grade,909044,19c5471b66eb08e4ba1790a5d56c1b40923fa1ecfd85ed...,SINGLETON,1,NaN,...,SHA_19c5471b66eb08e4ba1790a5d56c1b40923fa1ecfd...,train,SHA_SINGLETON_GROUP,RETAINED_QC_CLEAN,False,SHA256_SINGLE_IMAGE_GROUP,Legacy audit fields preserved for traceability...,Apple_1st grade,/content/fruit_freshness_cache/images/19c5471b...,/content/drive/MyDrive/Fruit_Freshness_Researc...
4,AFruitDB,IMG_20230930_172439.jpg,Fruits Grade/Fruit grade/Apple/1st grade/IMG_2...,Apple,1st grade,1140016,1d791349b86caa51efe1adf9c62cc70459498a4c909390...,SINGLETON,1,NaN,...,SHA_1d791349b86caa51efe1adf9c62cc70459498a4c90...,train,SHA_SINGLETON_GROUP,RETAINED_QC_CLEAN,False,SHA256_SINGLE_IMAGE_GROUP,Legacy audit fields preserved for traceability...,Apple_1st grade,/content/fruit_freshness_cache/images/1d791349...,/content/drive/MyDrive/Fruit_Freshness_Researc...



Value counts for 'grade':
grade
1st grade    210
2nd grade    210
3rd grade    210
Name: count, dtype: int64

Value counts for 'class_name':
class_name
Apple_1st grade            30
Apple_2nd grade            30
Apple_3rd grade            30
Banana_1st grade           30
Banana_2nd grade           30
Banana_3rd grade           30
Burmese_grape_1st grade    30
Burmese_grape_2nd grade    30
Burmese_grape_3rd grade    30
Jujube_1st grade           30
Jujube_2nd grade           30
Jujube_3rd grade           30
Mango_1st grade            30
Mango_2nd grade            30
Mango_3rd grade            30
Papayas_1st grade          30
Papayas_2nd grade          30
Papayas_3rd grade          30
Tomatoes_1st grade         30
Tomatoes_2nd grade         30
Tomatoes_3rd grade         30
Name: count, dtype: int64

=== CALIBRATION SELECTION EVIDENCE ===
{
  "audit_name": "calibration_subset_selection",
  "version": 1,
  "timestamp_utc": "2026-11-26T17:16:41.241243+00:00",
  "source_training_manifest": 

In [10]:
# ============================================================
# ANDROID EXPORT — CELL 11
# CPU ONLY
# Purpose: Verify that the 630 selected calibration image paths
# are accessible before calibration begins.
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Fruit_Freshness_Research")

CALIBRATION_CSV = (
    ROOT / "experiments" / "baseline_mobilenetv3_small"
    / "quantization" / "calibration_train_manifest_630_v1.csv"
)

assert CALIBRATION_CSV.exists(), "Calibration manifest missing."

df_cal = pd.read_csv(CALIBRATION_CSV)

# Prefer the previously audited cache paths for efficient access.
path_column = "cache_path"

assert path_column in df_cal.columns, (
    f"Expected path column '{path_column}' not found."
)

paths = [Path(str(p)) for p in df_cal[path_column]]

exists_flags = [p.is_file() for p in paths]

missing_paths = [
    str(p) for p, exists in zip(paths, exists_flags) if not exists
]

print("Calibration samples listed:", len(df_cal))
print("Image paths checked:", len(paths))
print("Existing files:", sum(exists_flags))
print("Missing files:", len(missing_paths))

if missing_paths:
    print("\nFirst missing paths:")
    for p in missing_paths[:20]:
        print(" -", p)

# Check path uniqueness
unique_paths = len(set(map(str, paths)))

print("\nUnique image paths:", unique_paths)
print("Duplicate path entries:", len(paths) - unique_paths)

# Class-wise path availability
df_cal["path_exists"] = exists_flags

print("\nPath availability by class:")
class_summary = (
    df_cal.groupby("class_name")["path_exists"]
    .agg(["count", "sum"])
)
class_summary["missing"] = (
    class_summary["count"] - class_summary["sum"]
)
display(class_summary)

print("\nCELL 11 — CALIBRATION PATH ACCESSIBILITY CHECK COMPLETE")


Calibration samples listed: 630
Image paths checked: 630
Existing files: 0
Missing files: 630

First missing paths:
 - /content/fruit_freshness_cache/images/0a3850fd7f0258c769cede51f9be4b2298be73c55490ec77f0adcb51b8077baf.jpg
 - /content/fruit_freshness_cache/images/0f0a7ced697563e9b2ad78510e00e468182e74640d44d63aa8078f3b9f6ee154.jpg
 - /content/fruit_freshness_cache/images/14e34a5184960b7b12c5d2435bfef24d426614c80dc1717f3ecdc731bc777864.jpg
 - /content/fruit_freshness_cache/images/19c5471b66eb08e4ba1790a5d56c1b40923fa1ecfd85ed0765f1fc64b51560a7.jpg
 - /content/fruit_freshness_cache/images/1d791349b86caa51efe1adf9c62cc70459498a4c909390a570589641eefee38d.jpg
 - /content/fruit_freshness_cache/images/46cdfb083c5841bcf8bf7dbe4febdc90fdabf029f015667399431fe1889dbca0.jpg
 - /content/fruit_freshness_cache/images/48caebaf012c7bd9883e5afb6d0845d1023efbc5ea521a6a60084ed47e7c9543.jpg
 - /content/fruit_freshness_cache/images/493e73a85328290e84fd96b05c8baf5fa48c6241b5d88c8a01ac6df7f8c84605.jpg
 - /

,count,sum,missing
class_name,,,
Apple_1st grade,30,0,30
Apple_2nd grade,30,0,30
Apple_3rd grade,30,0,30
Banana_1st grade,30,0,30
Banana_2nd grade,30,0,30
Banana_3rd grade,30,0,30
Burmese_grape_1st grade,30,0,30
Burmese_grape_2nd grade,30,0,30
Burmese_grape_3rd grade,30,0,30



CELL 11 — CALIBRATION PATH ACCESSIBILITY CHECK COMPLETE


In [11]:
# ============================================================
# ANDROID EXPORT — CELL 12
# CPU ONLY
# Purpose: Check Drive-backed image paths in calibration manifest.
# No images are decoded or loaded into memory.
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/Fruit_Freshness_Research")

CALIBRATION_CSV = (
    ROOT / "experiments" / "baseline_mobilenetv3_small"
    / "quantization" / "calibration_train_manifest_630_v1.csv"
)

df_cal = pd.read_csv(CALIBRATION_CSV)

path_column = "calibration_image_path"

assert path_column in df_cal.columns, (
    f"Column not found: {path_column}"
)

drive_paths = [
    Path(str(p)) for p in df_cal[path_column]
]

exists_flags = [p.is_file() for p in drive_paths]

missing_paths = [
    str(p) for p, exists in zip(drive_paths, exists_flags)
    if not exists
]

print("Calibration samples:", len(df_cal))
print("Drive paths checked:", len(drive_paths))
print("Existing Drive images:", sum(exists_flags))
print("Missing Drive images:", len(missing_paths))
print("Unique paths:", len(set(map(str, drive_paths))))

if missing_paths:
    print("\nFirst missing Drive paths:")
    for p in missing_paths[:10]:
        print(" -", p)

df_cal["drive_path_exists"] = exists_flags

print("\nAvailability by class:")
availability = df_cal.groupby("class_name")["drive_path_exists"].agg(
    total="count",
    available="sum"
)
availability["missing"] = availability["total"] - availability["available"]
display(availability)

print("\nCELL 12 — DRIVE CALIBRATION IMAGE PATH CHECK COMPLETE")

Calibration samples: 630
Drive paths checked: 630
Existing Drive images: 630
Missing Drive images: 0
Unique paths: 630

Availability by class:


,total,available,missing
class_name,,,
Apple_1st grade,30,30,0
Apple_2nd grade,30,30,0
Apple_3rd grade,30,30,0
Banana_1st grade,30,30,0
Banana_2nd grade,30,30,0
Banana_3rd grade,30,30,0
Burmese_grape_1st grade,30,30,0
Burmese_grape_2nd grade,30,30,0
Burmese_grape_3rd grade,30,30,0



CELL 12 — DRIVE CALIBRATION IMAGE PATH CHECK COMPLETE


In [12]:

# CELL 13 — SHA-256 INTEGRITY CHECK OF CALIBRATION IMAGES
# CPU only: no GPU, training, model inference, or test-set access.

import hashlib
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

CALIBRATION_MANIFEST = (
    "/content/drive/MyDrive/Fruit_Freshness_Research/"
    "experiments/baseline_mobilenetv3_small/"
    "quantization/calibration_train_manifest_630_v1.csv"
)

df_cal = pd.read_csv(CALIBRATION_MANIFEST)

# Verify required columns
required_columns = ["calibration_image_path", "sha256", "class_name"]
missing_columns = [c for c in required_columns if c not in df_cal.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

def calculate_sha256(file_path):
    sha256_hash = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256_hash.update(chunk)
    return sha256_hash.hexdigest()

print("Computing SHA-256 for 630 calibration images...")
actual_hashes = []

for path in tqdm(df_cal["calibration_image_path"], total=len(df_cal)):
    actual_hashes.append(calculate_sha256(path))

df_cal["actual_sha256"] = actual_hashes
df_cal["sha256_match"] = (
    df_cal["sha256"].astype(str).str.lower()
    == df_cal["actual_sha256"].astype(str).str.lower()
)

total = len(df_cal)
matched = int(df_cal["sha256_match"].sum())
mismatched = total - matched

print("\n--- CALIBRATION IMAGE INTEGRITY RESULTS ---")
print(f"Images checked: {total}")
print(f"SHA-256 matches: {matched}")
print(f"SHA-256 mismatches: {mismatched}")
print(f"Integrity rate: {matched / total * 100:.2f}%")

# Report mismatches explicitly
if mismatched > 0:
    print("\nWARNING — Hash mismatches detected:")
    display(
        df_cal.loc[
            ~df_cal["sha256_match"],
            ["class_name", "filename", "calibration_image_path",
             "sha256", "actual_sha256"]
        ]
    )
else:
    print("\nPASS — All calibration images match their recorded SHA-256 hashes.")

# Per-class integrity summary
class_integrity = (
    df_cal.groupby("class_name")["sha256_match"]
    .agg(total="count", matched="sum")
)
class_integrity["mismatched"] = class_integrity["total"] - class_integrity["matched"]

display(class_integrity)

print("\nCELL 13 — CALIBRATION SHA-256 INTEGRITY CHECK COMPLETE")


Computing SHA-256 for 630 calibration images...


  0%|          | 0/630 [00:00<?, ?it/s]


--- CALIBRATION IMAGE INTEGRITY RESULTS ---
Images checked: 630
SHA-256 matches: 630
SHA-256 mismatches: 0
Integrity rate: 100.00%

PASS — All calibration images match their recorded SHA-256 hashes.


,total,matched,mismatched
class_name,,,
Apple_1st grade,30,30,0
Apple_2nd grade,30,30,0
Apple_3rd grade,30,30,0
Banana_1st grade,30,30,0
Banana_2nd grade,30,30,0
Banana_3rd grade,30,30,0
Burmese_grape_1st grade,30,30,0
Burmese_grape_2nd grade,30,30,0
Burmese_grape_3rd grade,30,30,0



CELL 13 — CALIBRATION SHA-256 INTEGRITY CHECK COMPLETE


In [13]:

# CELL 14 — CALIBRATION IMAGE DECODE & PREPROCESSING VALIDATION
# CPU only. No GPU, model inference, training, quantization, validation, or test access.

from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageFile
import torch
from torchvision import transforms
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = False

CALIBRATION_MANIFEST = (
    "/content/drive/MyDrive/Fruit_Freshness_Research/"
    "experiments/baseline_mobilenetv3_small/"
    "quantization/calibration_train_manifest_630_v1.csv"
)

df_cal = pd.read_csv(CALIBRATION_MANIFEST)

# Match the baseline model's evaluation preprocessing
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Deterministic validation sample: first 3 records per class
sample_df = (
    df_cal.sort_values(["class_name", "filename"])
    .groupby("class_name", group_keys=False)
    .head(3)
    .reset_index(drop=True)
)

print(f"Images selected for decode/preprocessing validation: {len(sample_df)}")
print(f"Classes represented: {sample_df['class_name'].nunique()}")

results = []
tensor_shapes = []
tensor_dtypes = []
all_finite = True

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    image_path = Path(row["calibration_image_path"])

    with Image.open(image_path) as img:
        original_format = img.format
        original_mode = img.mode
        original_size = img.size

        # Explicit RGB conversion, consistent with model input requirements
        img_rgb = img.convert("RGB")
        tensor = preprocess(img_rgb)

    finite = bool(torch.isfinite(tensor).all().item())
    all_finite = all_finite and finite

    tensor_shapes.append(tuple(tensor.shape))
    tensor_dtypes.append(str(tensor.dtype))

    results.append({
        "class_name": row["class_name"],
        "filename": row["filename"],
        "original_format": original_format,
        "original_mode": original_mode,
        "original_width": original_size[0],
        "original_height": original_size[1],
        "tensor_shape": str(tuple(tensor.shape)),
        "tensor_dtype": str(tensor.dtype),
        "tensor_finite": finite,
        "tensor_min": float(tensor.min()),
        "tensor_max": float(tensor.max()),
        "tensor_mean": float(tensor.mean())
    })

df_decode = pd.DataFrame(results)

# Summary
print("\n--- IMAGE DECODE & PREPROCESSING SUMMARY ---")
print(f"Images successfully decoded: {len(df_decode)} / {len(sample_df)}")
print(f"Unique classes represented: {df_decode['class_name'].nunique()}")
print(f"All tensors have expected shape: {set(tensor_shapes) == {(3, 224, 224)}}")
print(f"All tensors are float32: {set(tensor_dtypes) == {'torch.float32'}}")
print(f"All tensor values finite: {all_finite}")
print(f"Observed image formats: {sorted(df_decode['original_format'].dropna().unique())}")
print(f"Observed original image modes: {sorted(df_decode['original_mode'].dropna().unique())}")

display(
    df_decode.groupby("class_name").agg(
        images_checked=("filename", "count"),
        all_finite=("tensor_finite", "all"),
        min_tensor_value=("tensor_min", "min"),
        max_tensor_value=("tensor_max", "max")
    )
)

if (
    len(df_decode) == 63
    and df_decode["class_name"].nunique() == 21
    and all(shape == (3, 224, 224) for shape in tensor_shapes)
    and all(dtype == "torch.float32" for dtype in tensor_dtypes)
    and all_finite
):
    print("\nPASS — Image decoding and preprocessing validation succeeded.")
else:
    print("\nWARNING — Validation did not meet all expected conditions.")
    display(df_decode)

print("\nCELL 14 — IMAGE PREPROCESSING VALIDATION COMPLETE")


Images selected for decode/preprocessing validation: 63
Classes represented: 21


  0%|          | 0/63 [00:00<?, ?it/s]


--- IMAGE DECODE & PREPROCESSING SUMMARY ---
Images successfully decoded: 63 / 63
Unique classes represented: 21
All tensors have expected shape: True
All tensors are float32: True
All tensor values finite: True
Observed image formats: ['JPEG']
Observed original image modes: ['RGB']


,images_checked,all_finite,min_tensor_value,max_tensor_value
class_name,,,,
Apple_1st grade,3,True,-1.998031,1.974912
Apple_2nd grade,3,True,-1.741159,2.640000
Apple_3rd grade,3,True,-1.563025,2.552854
Banana_1st grade,3,True,-1.948179,2.248908
Banana_2nd grade,3,True,-1.804444,2.248908
Banana_3rd grade,3,True,-1.861033,1.942832
Burmese_grape_1st grade,3,True,-1.706910,2.008403
Burmese_grape_2nd grade,3,True,-1.552787,2.413421
Burmese_grape_3rd grade,3,True,-1.703081,2.448279



PASS — Image decoding and preprocessing validation succeeded.

CELL 14 — IMAGE PREPROCESSING VALIDATION COMPLETE


In [17]:

# CELL 15 — PT2E / TORCHAO / EXECUTORCH API INSPECTION
# CPU only. Inspection-only; no model loading or quantization.

import inspect
import torch
import torchao
import executorch

from torchao.quantization.pt2e.quantize_pt2e import (
    prepare_pt2e,
    convert_pt2e
)
from torchao.quantization.pt2e.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer,
    get_symmetric_quantization_config
)
from executorch.backends.xnnpack.partition.xnnpack_partitioner import (
    XnnpackPartitioner
)

print("=" * 70)
print("ENVIRONMENT VERSIONS")
print("=" * 70)
print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)
print("ExecuTorch:", executorch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Workflow will use CPU: True")

print("\n" + "=" * 70)
print("PT2E API SIGNATURES")
print("=" * 70)

for obj in [prepare_pt2e, convert_pt2e]:
    print(f"\n{obj.__name__}")
    print("Signature:", inspect.signature(obj))
    print("Module:", obj.__module__)

print("\n" + "=" * 70)
print("XNNPACK QUANTIZER API")
print("=" * 70)

for obj in [XNNPACKQuantizer, get_symmetric_quantization_config]:
    print(f"\n{obj.__name__}")
    try:
        print("Signature:", inspect.signature(obj))
    except Exception as e:
        print("Signature unavailable:", repr(e))

print("\n" + "=" * 70)
print("XNNPACK PARTITIONER API")
print("=" * 70)

print("Class:", XnnpackPartitioner)
try:
    print("Constructor signature:", inspect.signature(XnnpackPartitioner))
except Exception as e:
    print("Constructor signature unavailable:", repr(e))

print("\nAvailable relevant XNNPACK quantizer methods:")
for name in dir(XNNPACKQuantizer):
    if any(k in name.lower() for k in ["config", "operator", "global", "set"]):
        print(" -", name)

print("\n" + "=" * 70)
print("EXECUTORCH EXPORT API")
print("=" * 70)

import executorch.exir as exir

print("executorch.exir available:", True)
for name in ["to_edge", "EdgeProgramManager"]:
    obj = getattr(exir, name, None)
    print(f"\n{name}:")
    print("Available:", obj is not None)
    if obj is not None:
        try:
            print("Signature:", inspect.signature(obj))
        except Exception as e:
            print("Signature unavailable:", repr(e))

print("\nCELL 15 — API INSPECTION COMPLETE")


ModuleNotFoundError: No module named 'executorch'

In [18]:

# CELL 15A — EXECUTORCH ENVIRONMENT DIAGNOSTIC
# CPU only; no package installation, model loading, or training.

import sys
import importlib.util
import subprocess

print("=" * 60)
print("CURRENT PYTHON KERNEL")
print("=" * 60)
print("Python executable:", sys.executable)
print("Python version:", sys.version)

print("\n" + "=" * 60)
print("PACKAGE IMPORT CHECK")
print("=" * 60)

packages = ["torch", "torchao", "executorch", "torchvision"]

for package in packages:
    spec = importlib.util.find_spec(package)
    print(f"{package}: {'FOUND' if spec is not None else 'NOT FOUND'}")
    if spec is not None:
        print("  Module location:", spec.origin)

print("\n" + "=" * 60)
print("PIP PACKAGE STATUS")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "-m", "pip", "show",
     "executorch", "torchao", "torch"],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("PIP diagnostic messages:")
    print(result.stderr)

print("\n" + "=" * 60)
print("CURRENT TORCH VERSION")
print("=" * 60)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
except Exception as e:
    print("PyTorch import error:", repr(e))

print("\nCELL 15A — ENVIRONMENT DIAGNOSTIC COMPLETE")


CURRENT PYTHON KERNEL
Python executable: /usr/bin/python3
Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]

PACKAGE IMPORT CHECK
torch: FOUND
  Module location: /usr/local/lib/python3.13/dist-packages/torch/__init__.py
torchao: FOUND
  Module location: /usr/local/lib/python3.13/dist-packages/torchao/__init__.py
executorch: NOT FOUND
torchvision: FOUND
  Module location: /usr/local/lib/python3.13/dist-packages/torchvision/__init__.py

PIP PACKAGE STATUS
Name: torchao
Version: 0.10.0
Summary: Package for applying ao techniques to GPU models
Home-page: https://github.com/pytorch/ao
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: 
Required-by: 
---
Name: torch
Version: 2.14.1+cpu
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License-Expression: Apache-2.0 AND Apache-2.0 WITH LLVM-exception AND BSD-2-C

In [19]:

import sys
import torch
import torchao
import importlib.util
import importlib.metadata as metadata

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)
print("CUDA available:", torch.cuda.is_available())

for package in ["torchvision", "executorch"]:
    print(
        f"{package}:",
        metadata.version(package)
        if importlib.util.find_spec(package) is not None
        else "NOT IMPORTABLE"
    )

print("\nEnvironment verification complete.")



Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Python executable: /usr/bin/python3
PyTorch: 2.14.1+cpu
TorchAO: 0.10.0
CUDA available: False
torchvision: 0.29.1+cpu
executorch: NOT IMPORTABLE

Environment verification complete.


In [20]:

# CELL 16 — PACKAGE COMPATIBILITY PRECHECK
# CPU only. Read-only diagnostic. No installation or model loading.

import sys
import subprocess
import importlib.metadata as md

print("Python executable:", sys.executable)
print("\n--- Installed package versions ---")

for package in ["torch", "torchvision", "torchao", "executorch"]:
    try:
        dist = md.distribution(package)
        print(f"\n{package}: {dist.version}")
        requirements = dist.metadata.get_all("Requires-Dist") or []
        for requirement in requirements:
            if any(k in requirement.lower() for k in ["torch", "numpy", "python"]):
                print("  Requirement:", requirement)
    except md.PackageNotFoundError:
        print(f"\n{package}: NOT INSTALLED")

print("\n--- Pip check ---")
result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)
print("Exit code:", result.returncode)
print(result.stdout or "(No dependency conflicts reported)")
print(result.stderr or "")

print("\nCELL 16 — PACKAGE COMPATIBILITY PRECHECK COMPLETE")


Python executable: /usr/bin/python3

--- Installed package versions ---

torch: 2.14.1+cpu

torchvision: 0.29.1+cpu
  Requirement: numpy
  Requirement: torch (>=2.14.0)

torchao: 0.10.0

executorch: NOT INSTALLED

--- Pip check ---
Exit code: 1
ipython 7.34.0 requires jedi, which is not installed.



CELL 16 — PACKAGE COMPATIBILITY PRECHECK COMPLETE


In [21]:

# CELL 17 — INSTALL EXECUTORCH
# Install the missing Android export package.

import sys
import subprocess

print("Installing ExecuTorch into the current Colab Python environment...")
print("Python:", sys.executable)

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "executorch"],
    check=False
)

if result.returncode == 0:
    print("\nInstallation command completed successfully.")
else:
    print(f"\nInstallation failed with exit code: {result.returncode}")

print("\nCELL 17 COMPLETE")


Installing ExecuTorch into the current Colab Python environment...
Python: /usr/bin/python3

Installation command completed successfully.

CELL 17 COMPLETE


In [22]:

# CELL 18 — VERIFY EXECUTORCH INSTALLATION
# CPU only. Verification; no training or model inference.

import sys
import importlib
import importlib.metadata as md

print("Python:", sys.version)
print("Executable:", sys.executable)

print("\n--- Package versions ---")
for pkg in ["torch", "torchvision", "torchao", "executorch"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except Exception as e:
        print(f"{pkg}: unavailable ({e})")

print("\n--- Import tests ---")
for module_name in [
    "torch",
    "torchao",
    "executorch",
    "executorch.exir",
]:
    try:
        module = importlib.import_module(module_name)
        print(f"PASS: {module_name}")
    except Exception as e:
        print(f"FAIL: {module_name} -> {type(e).__name__}: {e}")

print("\n--- PT2E API tests ---")
for module_name, symbols in [
    ("torchao.quantization.pt2e.quantize_pt2e",
     ["prepare_pt2e", "convert_pt2e"]),
    ("torchao.quantization.pt2e.quantizer.xnnpack_quantizer",
     ["XNNPACKQuantizer", "get_symmetric_quantization_config"]),
    ("executorch.backends.xnnpack.partition.xnnpack_partitioner",
     ["XnnpackPartitioner"]),
]:
    try:
        module = importlib.import_module(module_name)
        for symbol in symbols:
            print(f"{symbol}: {'AVAILABLE' if hasattr(module, symbol) else 'MISSING'}")
    except Exception as e:
        print(f"MODULE FAIL: {module_name}: {type(e).__name__}: {e}")

print("\nCELL 18 — INSTALLATION VERIFICATION COMPLETE")


Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Executable: /usr/bin/python3

--- Package versions ---
torch: 2.15.0
torchvision: 0.29.1+cpu
torchao: 0.18.0
executorch: 1.6.0

--- Import tests ---
PASS: torch
PASS: torchao
PASS: executorch
FAIL: executorch.exir -> AttributeError: '_OpNamespace' 'torchao' object has no attribute 'dequantize_affine'

--- PT2E API tests ---
MODULE FAIL: torchao.quantization.pt2e.quantize_pt2e: ImportError: cannot import name 'Granularity' from 'torchao.quantization' (/usr/local/lib/python3.13/dist-packages/torchao/quantization/__init__.py)
MODULE FAIL: torchao.quantization.pt2e.quantizer.xnnpack_quantizer: ImportError: cannot import name 'Granularity' from 'torchao.quantization' (/usr/local/lib/python3.13/dist-packages/torchao/quantization/__init__.py)
MODULE FAIL: executorch.backends.xnnpack.partition.xnnpack_partitioner: AttributeError: '_OpNamespace' 'torchao' object has no attribute 'dequantize_affine'

CELL 18 — INSTALLATION VERIFICATION C

In [23]:

# CELL 19 — PACKAGE/API MISMATCH DIAGNOSTIC
# CPU only. No installations, model loading, calibration, or export.

import sys
import subprocess
import importlib
import importlib.metadata as md

print("=" * 60)
print("INSTALLED DISTRIBUTION METADATA")
print("=" * 60)

for pkg in ["torch", "torchao", "executorch", "torchvision"]:
    try:
        dist = md.distribution(pkg)
        print(f"\n{pkg} == {dist.version}")
        for req in (dist.requires or []):
            if any(x in req.lower() for x in ["torch", "torchao"]):
                print("  dependency:", req)
    except Exception as e:
        print(pkg, "metadata error:", repr(e))

print("\n" + "=" * 60)
print("TORCHAO SYMBOL CHECK")
print("=" * 60)

import torchao
import torchao.quantization as tq

print("TorchAO location:", torchao.__file__)
print("torchao.quantization location:", tq.__file__)
print("Granularity exists:", hasattr(tq, "Granularity"))
print("Available quantization names containing 'gran':")
print([n for n in dir(tq) if "gran" in n.lower()])

print("\n" + "=" * 60)
"EXECUTORCH IMPORT DIAGNOSTIC"
print("=" * 60)

for module_name in [
    "executorch",
    "executorch.backends",
    "executorch.exir",
    "executorch.backends.xnnpack",
]:
    try:
        module = importlib.import_module(module_name)
        print("PASS:", module_name, getattr(module, "__file__", "(no file)"))
    except Exception as e:
        print("FAIL:", module_name, type(e).__name__, str(e))

print("\n" + "=" * 60)
print("PIP CHECK")
print("=" * 60)

r = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True, text=True
)
print("Return code:", r.returncode)
print(r.stdout or "No reported conflicts")
print(r.stderr or "")

print("\nCELL 19 — DIAGNOSTIC COMPLETE")


INSTALLED DISTRIBUTION METADATA

torch == 2.15.0

torchao == 0.18.0

executorch == 1.6.0
  dependency: pytorch-tokenizers
  dependency: torch==2.15.0
  dependency: torchao>=0.18.0

torchvision == 0.29.1+cpu
  dependency: torch (>=2.14.0)

TORCHAO SYMBOL CHECK
TorchAO location: /usr/local/lib/python3.13/dist-packages/torchao/__init__.py
torchao.quantization location: /usr/local/lib/python3.13/dist-packages/torchao/quantization/__init__.py
Granularity exists: False
Available quantization names containing 'gran':
['granularity']

PASS: executorch None
PASS: executorch.backends /usr/local/lib/python3.13/dist-packages/executorch/backends/__init__.py
FAIL: executorch.exir AttributeError '_OpNamespace' 'torchao' object has no attribute 'dequantize_affine'
FAIL: executorch.backends.xnnpack AttributeError '_OpNamespace' 'torchao' object has no attribute 'dequantize_affine'

PIP CHECK
Return code: 1
ipython 7.34.0 requires jedi, which is not installed.



CELL 19 — DIAGNOSTIC COMPLETE


In [24]:

# CELL 20 — PINNED EXECUTORCH CPU ENVIRONMENT
# Run in a fresh Colab runtime.
# CPU only; no dataset access, training, or model operations.

import sys
import subprocess

packages = [
    "torch==2.15.0",
    "torchao==0.18.0",
    "executorch==1.6.0",
]

cmd = [
    sys.executable, "-m", "pip", "install",
    "--upgrade",
    *packages,
    "--index-url", "https://download.pytorch.org/whl/cpu",
    "--extra-index-url", "https://pypi.org/simple",
]

print("Installing pinned CPU environment...")
print("Command:", " ".join(cmd))

result = subprocess.run(cmd)

if result.returncode == 0:
    print("\nINSTALL SUCCESS.")
    print("Restart the runtime before importing these packages.")
else:
    print("\nINSTALL FAILED.")
    print("Exit code:", result.returncode)

print("\nCELL 20 COMPLETE")



Installing pinned CPU environment...
Command: /usr/bin/python3 -m pip install --upgrade torch==2.15.0 torchao==0.18.0 executorch==1.6.0 --index-url https://download.pytorch.org/whl/cpu --extra-index-url https://pypi.org/simple

INSTALL SUCCESS.
Restart the runtime before importing these packages.

CELL 20 COMPLETE


In [1]:

# CELL 21 — VERIFY CLEAN PINNED ENVIRONMENT
# Run after runtime restart. CPU only; no training or model operations.

import sys
import importlib
import importlib.metadata as md

print("Python executable:", sys.executable)

print("\n--- Versions ---")
for pkg in ["torch", "torchao", "executorch", "torchvision"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except Exception as e:
        print(f"{pkg}: VERSION ERROR - {e}")

print("\n--- Core imports ---")
for module_name in ["torch", "torchao", "executorch"]:
    try:
        module = importlib.import_module(module_name)
        print(f"PASS: {module_name}")
    except Exception as e:
        print(f"FAIL: {module_name} | {type(e).__name__}: {e}")

print("\n--- PT2E/XNNPACK API imports ---")
checks = [
    ("torchao.quantization.pt2e.quantize_pt2e",
     ["prepare_pt2e", "convert_pt2e"]),
    ("torchao.quantization.pt2e.quantizer.xnnpack_quantizer",
     ["XNNPACKQuantizer", "get_symmetric_quantization_config"]),
    ("executorch.backends.xnnpack.partition.xnnpack_partitioner",
     ["XnnpackPartitioner"]),
]

for module_name, symbols in checks:
    try:
        module = importlib.import_module(module_name)
        missing = [s for s in symbols if not hasattr(module, s)]
        if not missing:
            print(f"PASS: {module_name}")
        else:
            print(f"PARTIAL: {module_name} | Missing: {missing}")
    except Exception as e:
        print(f"FAIL: {module_name} | {type(e).__name__}: {e}")

print("\nCELL 21 — ENVIRONMENT VERIFICATION COMPLETE")


Python executable: /usr/bin/python3

--- Versions ---
torch: 2.15.0+cpu
torchao: 0.18.0+cpu
executorch: 1.6.0+cpu
torchvision: 0.29.1+cpu

--- Core imports ---
PASS: torch


W1128 17:18:30.470000 46090 torch/utils/_pytree.py:646] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


PASS: torchao
PASS: executorch

--- PT2E/XNNPACK API imports ---
PASS: torchao.quantization.pt2e.quantize_pt2e
FAIL: torchao.quantization.pt2e.quantizer.xnnpack_quantizer | ModuleNotFoundError: No module named 'torchao.quantization.pt2e.quantizer.xnnpack_quantizer'
PASS: executorch.backends.xnnpack.partition.xnnpack_partitioner

CELL 21 — ENVIRONMENT VERIFICATION COMPLETE


In [3]:

# CELL 22 — LOCATE XNNPACK QUANTIZER APIs
# Read-only API inspection.

import importlib
import inspect

candidate_modules = [
    "executorch.backends.xnnpack.quantizer.xnnpack_quantizer",
    "executorch.backends.xnnpack.quantizer",
    "torchao.quantization.pt2e.quantizer",
]

found = {}

for module_name in candidate_modules:
    print(f"\nChecking: {module_name}")
    try:
        module = importlib.import_module(module_name)
        print("IMPORT: PASS")

        for symbol in [
            "XNNPACKQuantizer",
            "get_symmetric_quantization_config",
        ]:
            if hasattr(module, symbol):
                obj = getattr(module, symbol)
                found[symbol] = obj
                print(f"  FOUND: {symbol}")
                try:
                    print("  Signature:", inspect.signature(obj))
                except Exception:
                    pass
    except Exception as e:
        print("IMPORT: NOT AVAILABLE")
        print(type(e).__name__, str(e))

print("\n--- FINAL API SUMMARY ---")
for symbol in [
    "XNNPACKQuantizer",
    "get_symmetric_quantization_config",
]:
    print(f"{symbol}: {'AVAILABLE' if symbol in found else 'NOT FOUND'}")

print("\nCELL 22 COMPLETE")



Checking: executorch.backends.xnnpack.quantizer.xnnpack_quantizer
IMPORT: PASS
  FOUND: XNNPACKQuantizer
  Signature: () -> 'None'
  FOUND: get_symmetric_quantization_config
  Signature: (is_per_channel: 'bool' = False, is_qat: 'bool' = False, is_dynamic: 'bool' = False, act_qmin: 'int' = -128, act_qmax: 'int' = 127, weight_qmin: 'int' = -127, weight_qmax: 'int' = 127)

Checking: executorch.backends.xnnpack.quantizer
IMPORT: PASS

Checking: torchao.quantization.pt2e.quantizer
IMPORT: PASS

--- FINAL API SUMMARY ---
XNNPACKQuantizer: AVAILABLE
get_symmetric_quantization_config: AVAILABLE

CELL 22 COMPLETE


In [4]:

# CELL 23 — XNNPACK QUANTIZER CONFIGURATION INSPECTION
# Inspection only; no model, calibration, or conversion.

import inspect
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer,
    get_symmetric_quantization_config,
)

quantizer = XNNPACKQuantizer()
qconfig = get_symmetric_quantization_config()

print("=" * 60)
print("QUANTIZER")
print("=" * 60)
print("Type:", type(quantizer))
print("Constructor:", inspect.signature(XNNPACKQuantizer))

print("\n--- Configuration function ---")
print("Signature:", inspect.signature(get_symmetric_quantization_config))
print("Default config type:", type(qconfig))
print("Default config representation:")
print(repr(qconfig))

print("\n--- Quantizer methods ---")
for name in dir(quantizer):
    if not name.startswith("_"):
        attr = getattr(quantizer, name)
        if callable(attr):
            try:
                print(f"{name}{inspect.signature(attr)}")
            except Exception:
                print(name)

print("\n--- Quantizer configuration state ---")
try:
    print(vars(quantizer))
except Exception as e:
    print("State inspection unavailable:", repr(e))

print("\nCELL 23 — QUANTIZER INSPECTION COMPLETE")



QUANTIZER
Type: <class 'executorch.backends.xnnpack.quantizer.xnnpack_quantizer.XNNPACKQuantizer'>
Constructor: () -> 'None'

--- Configuration function ---
Signature: (is_per_channel: 'bool' = False, is_qat: 'bool' = False, is_dynamic: 'bool' = False, act_qmin: 'int' = -128, act_qmax: 'int' = 127, weight_qmin: 'int' = -127, weight_qmax: 'int' = 127)
Default config type: <class 'torchao.quantization.pt2e.quantizer.utils.QuantizationConfig'>
Default config representation:
QuantizationConfig(input_activation=QuantizationSpec(dtype=torch.int8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=-128, quant_max=127, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), output_activation=QuantizationSpec(dtype=torch.int8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=-128, quant_max=127, qscheme=

In [8]:
# CELL 24 — PREPARE PT2E INT8 QUANTIZATION
# CPU only. No training and no calibration yet.

import torch
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer,
    get_symmetric_quantization_config,
)
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e

# Configure XNNPACK INT8 quantization
quantizer = XNNPACKQuantizer()
quantizer.set_global(get_symmetric_quantization_config())

# Prepare the exported FP32 model for calibration
prepared_model = prepare_pt2e(
    exported_model.module(),
    quantizer
)

print("PT2E preparation successful")
print("Prepared model type:", type(prepared_model))
print("Graph nodes:", len(list(prepared_model.graph.nodes)))
print("Quantizer:", type(quantizer).__name__)
print("Calibration is NOT performed yet.")
print("Test partition remains untouched.")

NameError: name 'exported_model' is not defined

In [9]:
# CELL 24A — RESTORE FP32 EXPORT + PREPARE INT8
# CPU only. No training and no calibration.

import torch
from pathlib import Path
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer,
    get_symmetric_quantization_config,
)
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e

EXPORT_DIR = Path(
    "/content/drive/MyDrive/Fruit_Freshness_Research/experiments/"
    "baseline_mobilenetv3_small/android_export"
)

FP32_EXPORT_PATH = EXPORT_DIR / "baseline_fp32_exported_program.pt2"

# Reload the preserved FP32 export
exported_program = torch.export.load(str(FP32_EXPORT_PATH)

)
exported_model = exported_program

# Configure XNNPACK INT8 quantization
quantizer = XNNPACKQuantizer()
quantizer.set_global(get_symmetric_quantization_config())

# Prepare model for quantization calibration
prepared_model = prepare_pt2e(
    exported_model.module(),
    quantizer
)

print("=" * 55)
print("PT2E PREPARATION SUCCESSFUL")
print("=" * 55)
print("Export file exists:", FP32_EXPORT_PATH.exists())
print("Prepared model:", type(prepared_model).__name__)
print("Graph nodes:", len(list(prepared_model.graph.nodes)))
print("Quantizer:", type(quantizer).__name__)
print("Calibration performed: NO")
print("Training performed: NO")
print("Test set accessed: NO")

PT2E PREPARATION SUCCESSFUL
Export file exists: True
Prepared model: GraphModule
Graph nodes: 397
Quantizer: XNNPACKQuantizer
Calibration performed: NO
Training performed: NO
Test set accessed: NO


In [13]:
# CELL 25A — INT8 CALIBRATION (FIXED)
# CPU only. Training calibration data only.
# No gradient updates; do not call eval() on the exported model.

import pandas as pd
import torch
from PIL import Image
from pathlib import Path
from torchvision import transforms

CALIBRATION_CSV = (
    "/content/drive/MyDrive/Fruit_Freshness_Research/experiments/"
    "baseline_mobilenetv3_small/quantization/"
    "calibration_train_manifest_630_v1.csv"
)

df_cal = pd.read_csv(CALIBRATION_CSV)

assert len(df_cal) == 630
assert df_cal["class_name"].nunique() == 21
assert df_cal["split"].eq("train").all()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

calibration_count = 0

with torch.no_grad():
    for _, row in df_cal.iterrows():
        image_path = Path(row["calibration_image_path"])
        assert image_path.exists(), f"Missing image: {image_path}"

        image = Image.open(image_path).convert("RGB")
        tensor = transform(image).unsqueeze(0)

        prepared_model(tensor)
        calibration_count += 1

        if calibration_count % 100 == 0:
            print(f"Calibrated {calibration_count}/630 images")

assert calibration_count == 630

print("\nCALIBRATION COMPLETE")
print("Images calibrated:", calibration_count)
print("Classes:", df_cal["class_name"].nunique())
print("Train split only:", df_cal["split"].eq("train").all())
print("Test set accessed: NO")
print("Training/gradient updates: NO")

Calibrated 100/630 images
Calibrated 200/630 images
Calibrated 300/630 images
Calibrated 400/630 images
Calibrated 500/630 images
Calibrated 600/630 images

CALIBRATION COMPLETE
Images calibrated: 630
Classes: 21
Train split only: True
Test set accessed: NO
Training/gradient updates: NO


In [14]:
# CELL 26 — PT2E INT8 CONVERSION
# CPU only. No training or test-set access.

from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e

quantized_model = convert_pt2e(prepared_model)

print("=" * 55)
print("INT8 CONVERSION COMPLETE")
print("=" * 55)
print("Model type:", type(quantized_model).__name__)
print("Graph nodes:", len(list(quantized_model.graph.nodes)))

# Inspect quantization-related operations
quant_ops = [
    node for node in quantized_model.graph.nodes
    if "quantize" in str(node.target).lower()
    or "dequantize" in str(node.target).lower()
]

print("Quantization-related nodes:", len(quant_ops))
print("\nSample quantization operations:")
for node in quant_ops[:10]:
    print(" -", node.target)

print("\nTraining performed: NO")
print("Test set accessed: NO")

INT8 CONVERSION COMPLETE
Model type: GraphModule
Graph nodes: 521
Quantization-related nodes: 287

Sample quantization operations:
 - quantized_decomposed.dequantize_per_tensor.default
 - quantized_decomposed.dequantize_per_tensor.default
 - quantized_decomposed.dequantize_per_tensor.default
 - quantized_decomposed.dequantize_per_tensor.default
 - quantized_decomposed.dequantize_per_tensor.default
 - quantized_decomposed.dequantize_per_tensor.default
 - quantized_decomposed.dequantize_per_tensor.default
 - quantized_decomposed.dequantize_per_tensor.default
 - quantized_decomposed.dequantize_per_tensor.default
 - quantized_decomposed.dequantize_per_tensor.default

Training performed: NO
Test set accessed: NO


In [15]:
# CELL 27 — EXECUTORCH EXPORT API CHECK
# CPU only. Inspection only; no changes to model artifacts.

import inspect
from executorch.exir import to_edge
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

print("to_edge signature:")
print(inspect.signature(to_edge))

print("\nXNNPACK Partitioner:")
print(XnnpackPartitioner)

print("\nPartitioner signature:")
print(inspect.signature(XnnpackPartitioner))

print("\nINT8 model ready for export:", quantized_model is not None)
print("Quantized graph nodes:", len(list(quantized_model.graph.nodes)))
print("\nCELL 27 COMPLETE")

to_edge signature:
(*args: Any, **kwargs: Any) -> Any

XNNPACK Partitioner:
<class 'executorch.backends.xnnpack.partition.xnnpack_partitioner.XnnpackPartitioner'>

Partitioner signature:
(configs: Optional[List[Type[executorch.backends.xnnpack.partition.config.xnnpack_config.XNNPartitionerConfig]]] = None, config_precisions: Union[executorch.backends.xnnpack.partition.config.xnnpack_config.ConfigPrecisionType, List[executorch.backends.xnnpack.partition.config.xnnpack_config.ConfigPrecisionType], NoneType] = None, per_op_mode=False, verbose: bool = False, **kwargs)

INT8 model ready for export: True
Quantized graph nodes: 521

CELL 27 COMPLETE


In [18]:
# CELL 28 — XNNPACK EDGE + EXECUTORCH EXPORT
# CPU only. Synthetic input only; no dataset or test images used.

import torch
from pathlib import Path
from executorch.exir import to_edge
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

EXPORT_DIR = Path(
    "/content/drive/MyDrive/Fruit_Freshness_Research/experiments/"
    "baseline_mobilenetv3_small/android_export"
)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

INT8_PTE_PATH = EXPORT_DIR / "mobilenetv3_small_int8_xnnpack.pte"

# Use synthetic input to capture the converted INT8 graph
example_input = torch.randn(1, 3, 224, 224)

# Export the converted quantized graph
exported_int8 = torch.export.export(
    quantized_model,
    (example_input,),
    strict=False
)

# Convert to Edge dialect and delegate supported operators to XNNPACK
edge_program = to_edge(exported_int8)
edge_program = edge_program.to_backend(XnnpackPartitioner())

# Generate ExecuTorch program
executorch_program = edge_program.to_executorch()

# Save the binary artifact
with open(INT8_PTE_PATH, "wb") as f:
    f.write(executorch_program.buffer)

print("=" * 55)
print("EXECUTORCH EXPORT COMPLETE")
print("=" * 55)
print("Artifact:", INT8_PTE_PATH)
print("Exists:", INT8_PTE_PATH.exists())
print("Size (bytes):", INT8_PTE_PATH.stat().st_size)
print("Size (MB):", round(INT8_PTE_PATH.stat().st_size / (1024**2), 3))
print("Training performed: NO")
print("Test set accessed: NO")

DEPRECATION WARNING: You are using the deprecated 'to_edge() + to_backend()' workflow. Please consider migrating to 'to_edge_transform_and_lower()' for better error handling and optimization. 


EXECUTORCH EXPORT COMPLETE
Artifact: /content/drive/MyDrive/Fruit_Freshness_Research/experiments/baseline_mobilenetv3_small/android_export/mobilenetv3_small_int8_xnnpack.pte
Exists: True
Size (bytes): 1653204
Size (MB): 1.577
Training performed: NO
Test set accessed: NO


In [19]:
# CELL 29 — EXECUTORCH RUNTIME API CHECK
# CPU only. Inspection, no dataset access.

import inspect
import executorch.runtime as et_runtime

print("ExecuTorch runtime module:", et_runtime)
print("\nAvailable runtime symbols:")
print([name for name in dir(et_runtime) if not name.startswith("_")])

if hasattr(et_runtime, "Runtime"):
    print("\nRuntime class:", et_runtime.Runtime)
    try:
        print("Runtime.get signature:", inspect.signature(et_runtime.Runtime.get))
    except Exception as e:
        print("Signature unavailable:", e)

print("\nINT8 artifact exists:", INT8_PTE_PATH.exists())
print("INT8 artifact size:", INT8_PTE_PATH.stat().st_size, "bytes")
print("\nCELL 29 COMPLETE")

ExecuTorch runtime module: <module 'executorch.runtime' from '/usr/local/lib/python3.13/dist-packages/executorch/runtime/__init__.py'>

Available runtime symbols:
['Any', 'BackendRegistry', 'BinaryIO', 'Dict', 'ExecuTorchMethod', 'ExecuTorchProgram', 'ExecuTorchResult', 'List', 'Method', 'MethodMeta', 'ModuleType', 'OperatorRegistry', 'Optional', 'Path', 'Program', 'Runtime', 'Sequence', 'Set', 'Union', 'Verification', 'functools']

Runtime class: <class 'executorch.runtime.Runtime'>
Runtime.get signature: () -> 'Runtime'

INT8 artifact exists: True
INT8 artifact size: 1653204 bytes

CELL 29 COMPLETE


In [20]:
# CELL 30 — INSPECT EXECUTORCH LOADING API
# CPU only. No inference yet.

import inspect
from executorch.runtime import Runtime, Program, Method

runtime = Runtime.get()

print("Runtime instance:", type(runtime))

for obj, name in [
    (runtime, "Runtime instance"),
    (Program, "Program class"),
    (Method, "Method class"),
]:
    print(f"\n--- {name} ---")
    for attr_name in dir(obj):
        if any(key in attr_name.lower() for key in ["load", "execute", "method"]):
            attr = getattr(obj, attr_name, None)
            if callable(attr):
                try:
                    print(attr_name, inspect.signature(attr))
                except Exception:
                    print(attr_name, "(signature unavailable)")

print("\nCELL 30 COMPLETE")

Runtime instance: <class 'executorch.runtime.Runtime'>

--- Runtime instance ---
load_program (data: Union[bytes, bytearray, BinaryIO, pathlib._local.Path, str], *, verification: executorch.extension.pybindings._C.Verification = <Verification.InternalConsistency: 1>, enable_etdump: bool = False, debug_buffer_size: int = 0, data_path: Union[pathlib._local.Path, str, NoneType] = None, backend_options: Optional[Dict[str, Dict[str, Union[bool, int, str]]]] = None) -> executorch.runtime.Program

--- Program class ---
load_method (self, name: str) -> Optional[executorch.runtime.Method]

--- Method class ---
execute (self, inputs: Sequence[Any], clone_outputs: bool = True) -> Sequence[Any]

CELL 30 COMPLETE


In [21]:
# CELL 31 — EXECUTORCH RUNTIME SMOKE TEST
# CPU only. Synthetic input, no dataset/test access.

import torch
from executorch.runtime import Runtime

runtime = Runtime.get()

# Load the saved ExecuTorch program
program = runtime.load_program(str(INT8_PTE_PATH))

# Load the model's forward method
method = program.load_method("forward")
assert method is not None, "Forward method could not be loaded"

# Synthetic input matching the model input shape
torch.manual_seed(20261119)
dummy_input = torch.randn(1, 3, 224, 224)

# Run inference
outputs = method.execute([dummy_input])

print("=" * 55)
print("EXECUTORCH RUNTIME TEST")
print("=" * 55)
print("Artifact loaded: YES")
print("Forward method loaded: YES")
print("Inference executed: YES")
print("Output type:", type(outputs))
print("Output count:", len(outputs))

if len(outputs) > 0:
    out = outputs[0]
    print("Output shape:", getattr(out, "shape", "shape unavailable"))
    print("Output type:", type(out))
    try:
        print("All finite:", torch.isfinite(out).all().item())
    except Exception as e:
        print("Finite check unavailable:", str(e))

print("\nSynthetic input only: YES")
print("Test partition accessed: NO")
print("Training performed: NO")

EXECUTORCH RUNTIME TEST
Artifact loaded: YES
Forward method loaded: YES
Inference executed: YES
Output type: <class 'list'>
Output count: 1
Output shape: torch.Size([1, 21])
Output type: <class 'torch.Tensor'>
All finite: True

Synthetic input only: YES
Test partition accessed: NO
Training performed: NO


In [22]:
# CELL 32 — FINAL EXPORT ARTIFACT AUDIT
# CPU only. File inspection only; no training, inference, or dataset access.

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

EXPORT_DIR = Path(
    "/content/drive/MyDrive/Fruit_Freshness_Research/experiments/"
    "baseline_mobilenetv3_small/android_export"
)

fp32_path = EXPORT_DIR / "baseline_fp32_exported_program.pt2"
int8_path = EXPORT_DIR / "mobilenetv3_small_int8_xnnpack.pte"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert fp32_path.exists(), "FP32 export missing"
assert int8_path.exists(), "INT8 artifact missing"

fp32_size = fp32_path.stat().st_size
int8_size = int8_path.stat().st_size
reduction_pct = (1 - int8_size / fp32_size) * 100

audit = {
    "audit_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project": "Fruit Freshness Classification",
    "model": "MobileNetV3-Small",
    "num_classes": 21,
    "input_shape": [1, 3, 224, 224],
    "baseline_checkpoint_retrained": False,
    "calibration_images": 630,
    "calibration_classes": 21,
    "calibration_images_per_class": 30,
    "calibration_source": "training partition only",
    "validation_used_for_calibration": False,
    "test_used_for_calibration": False,
    "test_partition_accessed": False,
    "fp32_export": {
        "filename": fp32_path.name,
        "size_bytes": fp32_size,
        "sha256": sha256_file(fp32_path),
    },
    "int8_executorch_export": {
        "filename": int8_path.name,
        "size_bytes": int8_size,
        "sha256": sha256_file(int8_path),
        "runtime_load_success": True,
        "forward_method_load_success": True,
        "synthetic_inference_success": True,
        "synthetic_output_shape": [1, 21],
        "synthetic_output_finite": True,
    },
    "artifact_size_reduction_percent": round(reduction_pct, 2),
    "accuracy_evaluation_completed": False,
    "real_device_latency_evaluation_completed": False,
    "android_app_integration_completed": False,
}

audit_path = EXPORT_DIR / "final_int8_export_audit_v1.json"

assert not audit_path.exists(), "Audit file already exists; refusing to overwrite."

with open(audit_path, "w") as f:
    json.dump(audit, f, indent=2)

print("=" * 60)
print("FINAL ARTIFACT AUDIT PASSED")
print("=" * 60)
print(f"FP32 export: {fp32_size:,} bytes")
print(f"INT8 ExecuTorch: {int8_size:,} bytes")
print(f"Artifact size reduction: {reduction_pct:.2f}%")
print("FP32 SHA-256:", audit["fp32_export"]["sha256"])
print("INT8 SHA-256:", audit["int8_executorch_export"]["sha256"])
print("\nAudit saved:", audit_path)
print("Audit file exists:", audit_path.exists())
print("\nRuntime smoke test: PASS (verified in Cell 31)")
print("Training: NOT performed")
print("Test set: NOT accessed")
print("CELL 32 COMPLETE")

FINAL ARTIFACT AUDIT PASSED
FP32 export: 7,430,390 bytes
INT8 ExecuTorch: 1,653,204 bytes
Artifact size reduction: 77.75%
FP32 SHA-256: 60ebf8c481d6260791cecbe4b04678b4f18b65a788e437f6e7eca43e94c158a5
INT8 SHA-256: 333f73e5ebef78e3bb8055c783f848e78b786694e268bdf336d76e7bce0e208d

Audit saved: /content/drive/MyDrive/Fruit_Freshness_Research/experiments/baseline_mobilenetv3_small/android_export/final_int8_export_audit_v1.json
Audit file exists: True

Runtime smoke test: PASS (verified in Cell 31)
Training: NOT performed
Test set: NOT accessed
CELL 32 COMPLETE
